# Web services

So far every program ran on your own computer, started by you. Today: programs that other
programs talk to over the network.

- **Client:** sends a *request* ("give me passenger 1"). Your browser, a phone app, or Python.
- **Server:** runs all the time, waits for requests and sends back a *response*.

They talk **HTTP**. We start as a client, then write our own servers with three different
frameworks, all on the Titanic data from Nov 20:

| File (in `lectures/web/`) | Framework | Gives you |
|---|---|---|
| `titanic_api.py` | FastAPI | data (JSON) for other programs |
| `titanic_flask.py` | Flask | web pages (HTML) for people |
| `titanic_streamlit.py` | Streamlit | an interactive dashboard, almost no web knowledge needed |
| `train_model.py`, `titanic_features.py` | | trains the tree from Nov 20 once and saves it with `pickle` |

At the end: a database in a file with `sqlite3`.

## HTTP: requests and responses

A request has:

- a **method**: what to do. `GET` (give me), `POST` (here is something new), also `PUT`,
  `PATCH`, `DELETE`
- a **URL**: `https://jsonplaceholder.typicode.com/posts/1?some=thing`: where, which
  *path* (`/posts/1`), and optional *query parameters* after the `?`
- **headers**: extra information (which format, who's asking, ...)
- an optional **body**: data sent along, e.g. a filled-in form

A response has a **status code**, headers and a body. Status codes come in groups:

| Code | Meaning |
|---|---|
| 2xx | worked. `200 OK`, `201 Created` |
| 3xx | look elsewhere (redirect) |
| 4xx | *the client* did something wrong. `400 Bad Request`, `404 Not Found`, `422` (FastAPI: input doesn't fit) |
| 5xx | *the server* broke. `500 Internal Server Error` |

`jsonplaceholder.typicode.com` is a public test server with fake blog posts. `requests` is
the usual library for being a client in Python.

In [ ]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts/1")
print(response.status_code)
print(response.headers["Content-Type"])
print(response.text)

The body is text. This text is JSON (week 4), so `requests` can turn it into Python for us.
**Predict:** what type, and what's the title?

In [ ]:
post = response.json()
print(type(post))
print(post["title"])

There are 100 posts. **Predict:** the status code for post 300?

In [ ]:
response = requests.get("https://jsonplaceholder.typicode.com/posts/300")
print(response.status_code, response.json())

A missing post is not an exception in Python: the request worked, the answer is "not found".
Check the status code (or `response.raise_for_status()`, which raises for 4xx and 5xx).

In [ ]:
response.raise_for_status()

`POST` sends data in the body. A server that stored something new answers `201 Created`
(table above). There are 100 posts. **Predict:** which `id` does the new post get?

In [ ]:
new_post = {"userId": 1, "title": "Python is great", "body": "I really think so."}
response = requests.post("https://jsonplaceholder.typicode.com/posts", json=new_post)
print(response.status_code, response.json())

(It's a test server: it answers as if it had saved the post, but forgets it again.)

Query parameters: `requests` builds the part after the `?` for you, with `&` between two
parameters: `?userId=2&_limit=3`. The server sends the posts of user 2, at most 3:

In [ ]:
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts", params={"userId": 2, "_limit": 3}
)
print(response.url)
print([post["id"] for post in response.json()])

A list of dicts, straight from the network, is one step from a DataFrame:

In [ ]:
import pandas as pd

posts = pd.DataFrame(requests.get("https://jsonplaceholder.typicode.com/posts").json())
posts.groupby("userId").size().head()

## Our own server: FastAPI

Open `lectures/web/titanic_api.py` next to this notebook. Start it in a terminal, from the
repository folder:

```
uv run fastapi dev lectures/web/titanic_api.py
```

It keeps running (that's what a server does) and prints every request it gets. Then open
http://127.0.0.1:8000/docs in the browser: FastAPI writes this page from your code.

`127.0.0.1` (or `localhost`) is "this computer"; `8000` is the *port*, the door the server
listens at.

In [ ]:
API = "http://127.0.0.1:8000"

print(requests.get(API + "/").json())

Every `@app.get(...)` connects a URL to a function. FastAPI calls the function and turns what it
returns into JSON. The return type `-> PassengerInfo` is a class that lists the keys and their
types (Pydantic, more below); `/docs` shows it as the shape of the answer. In `/passengers/{passenger_id}`, the `{passenger_id}` part of the
path becomes the argument. **Predict** from the code in `titanic_api.py`: which keys come back
for passenger 1? And the status code for passenger 0 (the ids start at 1)?

In [ ]:
requests.get(API + "/passengers/1").json()

In [ ]:
response = requests.get(API + "/passengers/0")
print(response.status_code, response.json())

`raise HTTPException(status_code=404, ...)` stops the function and sends that status code.

The annotation `passenger_id: int` isn't just documentation here (it is in plain Python,
week 6): FastAPI converts the text from the URL to an `int`, and refuses when it can't.
**Predict** the status code (table above):

In [ ]:
response = requests.get(API + "/passengers/one")
print(response.status_code)
print(response.json())

Arguments that aren't in the path are **query parameters**. `= None` makes them optional.

`selected.query("Sex == @sex")` (pandas, nothing to do with query parameters) selects rows
like a mask from Nov 20: the condition is written as a string, with the column names as they
are, and `@sex` means the Python variable `sex`. It does the same as
`selected[selected["Sex"] == sex]`, without the nested brackets.
**Predict** the status codes:

In [ ]:
for params in [{}, {"sex": "female"}, {"sex": "female", "pclass": 3}, {"pclass": "first"}]:
    response = requests.get(API + "/survival", params=params)
    print(params, response.status_code, response.json())

**Predict:**

In [ ]:
response = requests.get(API + "/survival", params={"sex": "Female"})
print(response.status_code, response.json())

"No passengers match" is true, but is 404 the answer you'd want? The data says `female`, the
user typed `Female`. That's a design decision nobody wrote down, exactly what a review should
ask about.

### Sending data: `POST` and Pydantic

`/predict` gets a passenger in the body. `class PassengerFeatures(BaseModel)` describes
what the model needs to know about a passenger: FastAPI checks every request against it before
your function runs.
`Literal["female", "male"]` allows only these two values; `Field(ge=1, le=3)` means
"greater or equal 1, less or equal 3"; `= None` makes `age` optional.
**Predict** the status codes, also for the passenger without an age. (The survival chance
itself comes from the tree of Nov 20.)

In [ ]:
response = requests.post(API + "/predict", json={"sex": "female", "pclass": 3, "age": 30})
print(response.status_code, response.json())

In [ ]:
response = requests.post(API + "/predict", json={"sex": "male", "pclass": 1})
print(response.status_code, response.json())

**Predict:** status code, and how many problems does the answer list?

In [ ]:
response = requests.post(API + "/predict", json={"sex": "m", "pclass": 4, "age": -5})
print(response.status_code)
for problem in response.json()["detail"]:
    print(problem["loc"], problem["msg"])

The rules are in one place, and nonsense never reaches the model.

### Loading data and the model

`passengers = load_passengers()` and `pickle.load(...)` are at the top of the file, outside
any function. They run once when the server starts, not on every request: reading a CSV
for every request would make every answer slow. (And `pickle`: only load files you made
yourself, Wednesday.)

## Testing without a server

Starting a server for every test would be slow and fragile. FastAPI's `TestClient` sends
requests straight to the app, in the same Python process, no network:

In [ ]:
import sys

sys.path.insert(0, "web")  # so Python finds the files in lectures/web/

from fastapi.testclient import TestClient
from titanic_api import app

client = TestClient(app)
response = client.get("/survival", params={"sex": "female"})
print(response.status_code, response.json())

Same interface as `requests`. So tests look like this (pytest, week 2). **Predict:** which of
these tests can find a wrong survival rate?

In [ ]:
def test_survival_has_a_rate():
    response = client.get("/survival", params={"sex": "female"})
    assert response.status_code == 200
    assert "survival_rate" in response.json()


def test_survival_rate_of_women():
    response = client.get("/survival", params={"sex": "female"})
    assert response.json()["survival_rate"] == 0.742  # 233 of 314, counted by hand in pandas


def test_survival_unknown_class_is_rejected():
    assert client.get("/survival", params={"pclass": "first"}).status_code == 422


test_survival_has_a_rate()
test_survival_rate_of_women()
test_survival_unknown_class_is_rejected()

The first test passes as long as *some* number comes back, even a wrong one. Tests that only
check that a key exists are very common, especially in AI-written code. Keep that in mind for
Exercise 8.

### Things you'll see in other people's FastAPI code

- `async def` instead of `def`: lets the server do other work while a function waits (for a
  database, another server). Both work; for our apps plain `def` is fine.
- `@app.on_event("startup")`: an older way to run code when the server starts. It's
  deprecated; the current way is a `lifespan` function. Loading at the top of the file, as we
  do, also works.
- `response_model=...` on `@app.get`: the older way to say what we say with the return type
  (`-> PassengerInfo`). Both make FastAPI check what goes out and show it in `/docs`.
- `app = create_app(...)`: the app is built in a function, so tests can build one with
  different data.

## Flask: pages for people

FastAPI answers with data for other programs. Flask is older and more general; it's often
used to send whole **web pages**. Open `lectures/web/titanic_flask.py` and start it in a
second terminal:

```
uv run flask --app lectures/web/titanic_flask.py run --debug
```

Then open http://127.0.0.1:5000. The page is HTML with a *template*: `{{ result }}` and
`{% if %}` are filled in by Flask (with Jinja) before the page is sent.

A browser that submits the form sends a `POST` with *form data*, not JSON. From Python:

In [ ]:
FLASK = "http://127.0.0.1:5000"

page = requests.post(FLASK + "/", data={"sex": "female", "pclass": "1", "age": "30"}).text
print([line.strip() for line in page.splitlines() if "<strong>" in line])

**Predict:** FastAPI answered 422 for nonsense input. What does this Flask app do?

In [ ]:
response = requests.post(FLASK + "/", data={"sex": "female", "pclass": "first", "age": ""})
print(response.status_code)

`int("first")` raises a `ValueError` inside our function, and Flask turns any exception into
`500 Internal Server Error`: "the server broke", although the user typed something wrong.
In Flask, checking input is your job.

## Streamlit: a dashboard in a script

Streamlit is made for data people: no HTML, no routes. Open
`lectures/web/titanic_streamlit.py` and start it:

```
uv run streamlit run lectures/web/titanic_streamlit.py
```

The idea behind it: **every time someone moves a slider, the whole script runs again from
the top**, with the new values. **Predict:** how often does `script runs` appear in the
terminal after you move the age slider three times? And `loading the CSV`?

`@st.cache_data` remembers what a function returned, so the CSV is only read once (like
`@cache` on Nov 18).

**Predict:** the dashboard starts with all classes and ages 0 to 80. How many passengers does
it show?

In [ ]:
passengers = pd.read_csv("titanic.csv")
print(len(passengers))
print(len(passengers.query("0 <= Age <= 80")))

Any comparison with a missing age is False (Nov 20), so the dashboard quietly leaves out the
177 passengers whose age is unknown. Nothing on the page says so. Is that what the user
expects?

## A database in a file: `sqlite3`

A CSV is read completely, every time. A **database** stores tables so you can ask for exactly
the rows you need, and add or change rows without rewriting the file. Big systems run a
database server (PostgreSQL, MySQL). `sqlite3` is in the standard library and keeps a whole
database in one file: enough for most projects in this course.

You talk to it in **SQL**, its own small language.

In [ ]:
import sqlite3
from pathlib import Path

db_file = Path("titanic.db")
connection = sqlite3.connect(db_file)  # creates the file if it doesn't exist
passengers.to_sql("passengers", connection, if_exists="replace", index=False)

rows = connection.execute("SELECT Name, Age FROM passengers WHERE Pclass = 1 LIMIT 3").fetchall()
print(rows)

`SELECT` columns `FROM` a table `WHERE` a condition. Each row comes back as a tuple.
`GROUP BY Sex` makes one result row per value, like pandas' `groupby` on Nov 20, and `COUNT(*)`
and `AVG(...)` work on each group:

In [ ]:
query = "SELECT Sex, COUNT(*), AVG(Survived) FROM passengers GROUP BY Sex"
print(connection.execute(query).fetchall())

`GROUP BY` is pandas' `groupby`. For a DataFrame straight from SQL: `pd.read_sql_query`.

In [ ]:
pd.read_sql_query("SELECT Pclass, AVG(Fare) AS fare FROM passengers GROUP BY Pclass", connection)

### Values from users: always with `?`

A web app takes the name to look up from the user. It's tempting to build the query with an
f-string. **Predict** (f-strings, week 4): what query does the second call send to the database?

In [ ]:
def find_by_name(name):
    query = f"SELECT Name FROM passengers WHERE Name = '{name}'"
    return connection.execute(query).fetchall()


print(find_by_name("Braund, Mr. Owen Harris"))
print(len(find_by_name("x' OR '1'='1")))

The query is `... WHERE Name = 'x' OR '1'='1'`: the user's text became part of the SQL, and
`'1'='1'` is true for every row, so every passenger comes back.
That's **SQL injection**, one of the most common security holes on the web
(https://xkcd.com/327/). With `?`, the database gets the value separately, and it is only
ever a value:

In [ ]:
def find_by_name(name):
    return connection.execute("SELECT Name FROM passengers WHERE Name = ?", (name,)).fetchall()


print(find_by_name("Braund, Mr. Owen Harris"))
print(len(find_by_name("x' OR '1'='1")))

The same goes for pandas' `query` in `titanic_api.py`: `query("Sex == @sex")` hands over the
value separately. `query(f"Sex == '{sex}'")` would put the user's text into an expression
that pandas evaluates.

In [ ]:
connection.close()
db_file.unlink()  # remove the demo file again

## Which one for your project?

| | FastAPI | Flask | Streamlit |
|---|---|---|---|
| Answers with | data (JSON) | pages (HTML) or data | a finished dashboard |
| Checks input | automatically (types, Pydantic) | you do it | widgets only allow valid values |
| Testing | `TestClient`, easy | `app.test_client()`, easy | possible, harder |
| Good for | a service other programs use | a classic website | showing data and a model quickly |
| You need to know | HTTP, JSON | HTTP, HTML | almost nothing about the web |

All three are fine for the project. Many teams combine them: a FastAPI service for the data and
the model, and a Streamlit page that calls it with `requests`.